# PathTrace: full experiment grid on Colab

Runs every experiment behind the paper's tables and figures on a GPU, from the `pathtrace-rebuild` branch.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

Everything (data, checkpoints, results) is saved to Google Drive under `MyDrive/pathtrace`, and each step skips work that is already finished. If Colab disconnects, reconnect and run all cells again: the grid picks up where it stopped.

| Dataset | What runs | Why |
| --- | --- | --- |
| `junyi` (random negatives) | simulators, 4 baselines, 3 model variants × 5 seeds, RL × 5 seeds, RL sensitivity, DKT gain | Main results and ablations |
| `junyi-popneg` (popularity-sampled negatives) | baselines, pointer × 3 seeds, RL × 3 seeds, DKT gain | Harder candidate sets |
| `junyi-paper` (2,000 students, last 3 concepts each) | pointer × 3 seeds | Bridge to the paper's original setup |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Drive, code and dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/pathtrace'
REPO = '/content/AI-Project'
BRANCH = 'pathtrace-rebuild'
import os
os.makedirs(ROOT, exist_ok=True)

In [ ]:
import os
if os.path.exists(REPO):
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} https://github.com/VedanthParth/AI-Project.git {REPO}
!git -C {REPO} log --oneline -1
!pip install -q -e {REPO}/ml

## 2. Data (downloaded once into Drive; no Kaggle account needed)

In [ ]:
RAW = f'{ROOT}/data/raw/junyi'
if not os.path.exists(f'{RAW}/Log_Problem.csv'):
    os.makedirs(RAW, exist_ok=True)
    !curl -sL -o /content/junyi.zip https://www.kaggle.com/api/v1/datasets/download/junyiacademy/learning-activity-public-dataset-by-junyi-academy
    !unzip -q -o /content/junyi.zip -d {RAW} && rm /content/junyi.zip
!ls -la {RAW}

In [ ]:
PROC = f'{ROOT}/data/processed'
LOG = f'{RAW}/Log_Problem.csv'
datasets = {
    'junyi': '',
    'junyi-popneg': '--negatives popularity',
    'junyi-paper': '--windows last --max-students 2000',
}
for name, flags in datasets.items():
    if not os.path.exists(f'{PROC}/{name}/meta.json'):
        !python -m pathtrace prepare --log {LOG} --out {PROC}/{name} {flags}
    else:
        print(f'{name}: already prepared')

## 3. Experiment grids

`BATCH = 256` keeps the GPU busy; every final number in the paper should come from these runs, so report this batch size in the paper.

The main grid is the long step: each pointer-network run trains for up to 50 epochs, and there are 3 variants per seed. Watch the per-epoch time it prints in the first run to estimate the total. If Colab time is short, set `SEEDS = '0 1 2'` (3 seeds instead of 5) and skip the two optional grids. After a disconnect, run all cells again: finished runs are skipped.

In [ ]:
RUNS = f'{ROOT}/runs'
CONTENT = f'{RAW}/Info_Content.csv'
BATCH = 256
SEEDS = '0 1 2 3 4'   # use '0 1 2' if Colab time is short
COMMON = f'--content {CONTENT} --batch-size {BATCH} --device cuda'

In [ ]:
# Main results: 3 variants x SEEDS, RL per seed, sensitivity runs, DKT learning gain.
!python -m pathtrace grid --data {PROC}/junyi --runs {RUNS}/junyi {COMMON} --seeds {SEEDS}

In [ ]:
# Optional. Harder candidate sets: pointer network and RL, 3 seeds.
!python -m pathtrace grid --data {PROC}/junyi-popneg --runs {RUNS}/junyi-popneg {COMMON} --variants pointer --seeds 0 1 2 --no-sensitivity

In [ ]:
# Optional. The paper's original setup, for comparison with its reported numbers.
!python -m pathtrace grid --data {PROC}/junyi-paper --runs {RUNS}/junyi-paper {COMMON} --variants pointer --seeds 0 1 2 --no-rl --no-gain

## 4. Tables and figures for the paper

In [ ]:
for name in datasets:
    if os.path.exists(f'{RUNS}/{name}'):
        !python -m pathtrace report --data {PROC}/{name} --runs {RUNS}/{name} --out {ROOT}/results/{name}
!ls -R {ROOT}/results

The `results/` folder in Drive holds the tables (Markdown, LaTeX, CSV) and figures (PNG). The next cell shows the main ones here, so you can screenshot or download them.

In [ ]:
from IPython.display import Image, Markdown, display
out = f'{ROOT}/results/junyi'
display(Markdown(open(f'{out}/main_table.md').read()))
for fig in ['fig_methods.png', 'fig_training.png', 'fig_rl.png']:
    print(fig)
    display(Image(f'{out}/{fig}'))

## 5. App bundle from the full-data runs

Packs the seed-0 pointer network, its simulators and 300 held-out test students for the demo app. Download `bundle.zip` from `MyDrive/pathtrace`, unzip it, and replace the repo's `backend/bundle` folder with it.

In [ ]:
!python -m pathtrace export --data {PROC}/junyi --runs {RUNS}/junyi --checkpoint pointer-seed0 --content {CONTENT} --out /content/bundle
!cd /content && zip -qr bundle.zip bundle && cp bundle.zip {ROOT}/bundle.zip && ls -la {ROOT}/bundle.zip